# CO_dual_correction



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/Conformal_Oracle/blob/main/CO_dual_correction/CO_dual_correction.ipynb)



**Dual conformal prediction: one-sided VaR score + ES residual score → distribution-free correction**



Published in: *Calibrating the Oracle — Distribution-Free Conformal Risk Measures for LLM-Based Financial Forecasting*



Section: Conformal Calibration


In [ ]:
# Install dependencies (if running on Google Colab)

import sys

if 'google.colab' in sys.modules:

    !pip install -q numpy pandas matplotlib scipy


In [ ]:
import numpy as np

class DualConformalCalibrator:
    """Dual conformal calibrator for VaR and ES.

    Implements Algorithm 1 from 'Calibrating the Oracle':
    - VaR score: s_t^V = q_lo(t) - r(t)  (one-sided)
    - ES score: s_t^E = r(t) + ES(t)  (on violation days)
    """
    def __init__(self, alpha_var=0.01, alpha_es=0.025, f_cal=0.70):
        self.alpha_var = alpha_var
        self.alpha_es = alpha_es
        self.f_cal = f_cal

    def calibrate(self, samples, realized):
        T = len(samples)
        n_cal = int(T * self.f_cal)

        # Step 1: Raw quantiles
        q_lo = np.array([np.quantile(s[~np.isnan(s)], self.alpha_var) for s in samples])
        raw_var = -q_lo

        # ES: mean of lowest alpha fraction
        raw_es = np.array([
            -np.mean(np.sort(s[~np.isnan(s)])[:max(int(np.sum(~np.isnan(s)) * self.alpha_var), 1)])
            for s in samples
        ])

        # Step 2: VaR nonconformity scores (calibration set)
        s_V = q_lo[:n_cal] - realized[:n_cal]
        s_V_valid = s_V[~np.isnan(s_V)]
        q_level = min(np.ceil((len(s_V_valid) + 1) * (1 - self.alpha_var)) / len(s_V_valid), 1.0)
        q_hat_V = np.quantile(s_V_valid, q_level)

        # Step 3: ES residual scores (violation days only)
        s_E = []
        for t in range(n_cal):
            if realized[t] < -raw_var[t]:  # violation day
                s_E.append(realized[t] + raw_es[t])
        s_E = np.array(s_E)
        if len(s_E) > 0:
            q_level_E = min(np.ceil((len(s_E) + 1) * (1 - self.alpha_es)) / len(s_E), 1.0)
            q_hat_E = np.quantile(s_E, q_level_E)
        else:
            q_hat_E = 0.0

        # Step 4: Corrected estimates (test set)
        corrected_var = np.full(T, np.nan)
        corrected_es = np.full(T, np.nan)
        for t in range(n_cal, T):
            corrected_var[t] = raw_var[t] + q_hat_V
            corrected_es[t] = raw_es[t] + q_hat_E

        return {
            'q_hat_V': q_hat_V, 'q_hat_E': q_hat_E,
            'raw_var': raw_var, 'raw_es': raw_es,
            'corrected_var': corrected_var, 'corrected_es': corrected_es,
            'n_cal': n_cal,
        }

# ── Example ──
np.random.seed(42)
T = 500
samples = np.random.standard_t(df=5, size=(T, 1024)) * 0.01
realized = np.random.standard_t(df=5, size=T) * 0.01

cal = DualConformalCalibrator()
result = cal.calibrate(samples, realized)
print(f"Calibration set: {result['n_cal']} days")
print(f"q̂_V = {result['q_hat_V']:.6f}  (VaR correction threshold)")
print(f"q̂_E = {result['q_hat_E']:.6f}  (ES correction threshold)")
print(f"Mean raw VaR:       {np.nanmean(result['raw_var']):.6f}")
print(f"Mean corrected VaR: {np.nanmean(result['corrected_var']):.6f}")